# Row-Order Robustness Check for Modwheel Prefiltering (Colab)

This notebook addresses a key objection to row-ID wheel filtering:

> Are the results just an artifact of dataset row ordering?

We compare the same QOS-style wrapper pipeline under two conditions:

```text
original row order -> modwheel filter -> vectorize -> classifier
shuffled row order -> modwheel filter -> vectorize -> classifier
```

If results remain stable, the prefilter behavior is less likely to depend on accidental row ordering.

Guardrail: this is a robustness check for an adapter pattern. It does **not** claim QOS improvement, quantum advantage, or accuracy improvement.

In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS
from pre_oracle_filter import pre_oracle_candidates_by_name

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

## 1. Load raw 20 Newsgroups text stream

In [ ]:
RANDOM_STATE = 9423

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

raw_texts = np.array(dataset.data, dtype=object)
raw_y = np.array(dataset.target)
target_names = dataset.target_names
n_original = len(raw_texts)

n_original, target_names

## 2. Define wrapper pipeline

In [ ]:
def class_balance_l1_shift(y_subset, y_reference, n_classes):
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def evaluate_classifier(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    train_time = time.perf_counter() - t0
    pred = clf.predict(X_test)
    return {
        "train_time_seconds": train_time,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    }


def run_wrapper_pipeline(texts, y, order_label="original", wheel_name=None, y_reference=None):
    """Optional modwheel prefilter before vectorization and classifier sanity check."""
    if y_reference is None:
        y_reference = y

    n_before = len(texts)
    if wheel_name is None:
        keep_ids = np.arange(n_before)
        subset = "baseline"
    else:
        keep_ids = np.array(pre_oracle_candidates_by_name(np.arange(n_before), wheel_name), dtype=int)
        subset = wheel_name

    filtered_texts = list(texts[keep_ids])
    filtered_y = y[keep_ids]

    t_vec0 = time.perf_counter()
    vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
    X = vectorizer.fit_transform(filtered_texts)
    vectorize_time = time.perf_counter() - t_vec0

    eval_result = evaluate_classifier(X, filtered_y)

    return {
        "order_label": order_label,
        "subset": subset,
        "wheel": wheel_name if wheel_name is not None else "none",
        "n_original": n_before,
        "n_retained": len(filtered_texts),
        "retained_fraction": len(filtered_texts) / n_before,
        "reduction_fraction": 1 - len(filtered_texts) / n_before,
        "n_features": X.shape[1],
        "nnz": X.nnz,
        "vectorize_time_seconds": vectorize_time,
        "train_time_seconds": eval_result["train_time_seconds"],
        "accuracy": eval_result["accuracy"],
        "balanced_accuracy": eval_result["balanced_accuracy"],
        "class_balance_l1_shift": class_balance_l1_shift(filtered_y, y_reference, len(target_names)),
    }


run_wrapper_pipeline(raw_texts, raw_y, order_label="original", wheel_name="mod30")

## 3. Create shuffled row order

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(n_original)

shuffled_texts = raw_texts[perm]
shuffled_y = raw_y[perm]

# Same labels, different order.
np.array_equal(np.sort(shuffled_y), np.sort(raw_y)), perm[:10]

## 4. Run original-order and shuffled-order pipelines

In [ ]:
results = []

for order_label, texts, y_values in [
    ("original", raw_texts, raw_y),
    ("shuffled", shuffled_texts, shuffled_y),
]:
    # Baseline all rows.
    results.append(run_wrapper_pipeline(texts, y_values, order_label=order_label, wheel_name=None, y_reference=raw_y))
    # Wheel-filtered rows.
    for wheel_name in STANDARD_WHEELS:
        results.append(run_wrapper_pipeline(texts, y_values, order_label=order_label, wheel_name=wheel_name, y_reference=raw_y))

results_df = pd.DataFrame(results)
results_csv = data_dir / "07_row_order_robustness_results.csv"
results_df.to_csv(results_csv, index=False)
results_df

## 5. Build original-vs-shuffled delta table

In [ ]:
orig = results_df[results_df["order_label"] == "original"].set_index("subset")
shuf = results_df[results_df["order_label"] == "shuffled"].set_index("subset")
subsets = ["baseline", "mod30", "mod210", "mod2310"]

delta_rows = []
for subset in subsets:
    delta_rows.append({
        "subset": subset,
        "balanced_accuracy_original": orig.loc[subset, "balanced_accuracy"],
        "balanced_accuracy_shuffled": shuf.loc[subset, "balanced_accuracy"],
        "delta_balanced_accuracy": shuf.loc[subset, "balanced_accuracy"] - orig.loc[subset, "balanced_accuracy"],
        "vectorize_time_original": orig.loc[subset, "vectorize_time_seconds"],
        "vectorize_time_shuffled": shuf.loc[subset, "vectorize_time_seconds"],
        "delta_vectorize_time": shuf.loc[subset, "vectorize_time_seconds"] - orig.loc[subset, "vectorize_time_seconds"],
        "class_balance_l1_original": orig.loc[subset, "class_balance_l1_shift"],
        "class_balance_l1_shuffled": shuf.loc[subset, "class_balance_l1_shift"],
        "delta_class_balance_l1": shuf.loc[subset, "class_balance_l1_shift"] - orig.loc[subset, "class_balance_l1_shift"],
    })

delta_df = pd.DataFrame(delta_rows)
delta_csv = data_dir / "07_row_order_robustness_delta_summary.csv"
delta_df.to_csv(delta_csv, index=False)
delta_df

## 6. Figure 07a — balanced accuracy: original vs shuffled

In [ ]:
plot_order = ["baseline", "mod30", "mod210", "mod2310"]
orig_vals = delta_df.set_index("subset").loc[plot_order, "balanced_accuracy_original"]
shuf_vals = delta_df.set_index("subset").loc[plot_order, "balanced_accuracy_shuffled"]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(plot_order, orig_vals, marker="o", linewidth=2, label="original order")
ax.plot(plot_order, shuf_vals, marker="o", linewidth=2, label="shuffled order")
ax.set_title("Row-Order Robustness: Balanced Accuracy")
ax.set_xlabel("Subset")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()
fig_a_svg = fig_dir / "figure_07a_row_order_balanced_accuracy.svg"
fig_a_png = fig_dir / "figure_07a_row_order_balanced_accuracy.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png

## 7. Figure 07b — balanced accuracy delta

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.axhline(0, linestyle="--", linewidth=1)
ax.bar(delta_df["subset"], delta_df["delta_balanced_accuracy"])
ax.set_title("Shuffled Minus Original Balanced Accuracy")
ax.set_xlabel("Subset")
ax.set_ylabel("Δ balanced accuracy")
ax.grid(True, axis="y", alpha=0.35)
for i, value in enumerate(delta_df["delta_balanced_accuracy"]):
    ax.text(i, value, f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top")
fig.tight_layout()
fig_b_svg = fig_dir / "figure_07b_row_order_balanced_accuracy_delta.svg"
fig_b_png = fig_dir / "figure_07b_row_order_balanced_accuracy_delta.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png

## 8. Figure 07c — class-balance shift

In [ ]:
x = np.arange(len(plot_order))
width = 0.36
orig_balance = delta_df.set_index("subset").loc[plot_order, "class_balance_l1_original"]
shuf_balance = delta_df.set_index("subset").loc[plot_order, "class_balance_l1_shuffled"]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(x - width/2, orig_balance, width, label="original order")
ax.bar(x + width/2, shuf_balance, width, label="shuffled order")
ax.set_title("Row-Order Robustness: Class-Balance L1 Shift")
ax.set_xlabel("Subset")
ax.set_ylabel("L1 shift vs full dataset")
ax.set_xticks(x)
ax.set_xticklabels(plot_order)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()
fig.tight_layout()
fig_c_svg = fig_dir / "figure_07c_row_order_class_balance_shift.svg"
fig_c_png = fig_dir / "figure_07c_row_order_class_balance_shift.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png

## 9. Interpretation helper

In [ ]:
for _, row in delta_df.iterrows():
    print(f"{row['subset']}: shuffled - original balanced accuracy = {row['delta_balanced_accuracy']:+.4f}")

print("""
Interpretation:

- If shuffled and original results are close, row-ID filtering is less likely to depend on accidental dataset ordering.
- If deltas are large, row order may matter and the method should be framed as order-sensitive.
- Class-balance shift helps diagnose whether row order changes label distribution after filtering.

Guardrail:
This robustness check supports adapter interpretation only. It does not prove QOS improvement or quantum advantage.
""")

## 10. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/07_row_order_robustness_results.csv",
    "data/07_row_order_robustness_delta_summary.csv",
    "figures/figure_07a_row_order_balanced_accuracy.svg",
    "figures/figure_07b_row_order_balanced_accuracy_delta.svg",
    "figures/figure_07c_row_order_class_balance_shift.svg",
]:
    files.download(path)